In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [ ]:
df = pd.read_csv('/content/skill_builder_data.csv', encoding='latin1')
df.head()

,order_id,assignment_id,user_id,assistment_id,problem_id,original,correct,attempt_count,ms_first_response,tutor_mode,...,hint_count,hint_total,overlap_time,template_id,answer_id,answer_text,first_action,bottom_hint,opportunity,opportunity_original
0,33022537,277618,64525,33139,51424,1,1,1,32454,tutor,...,0,3,32454,30799,NaN,26,0,NaN,1,1.0
1,33022709,277618,64525,33150,51435,1,1,1,4922,tutor,...,0,3,4922,30799,NaN,55,0,NaN,2,2.0
2,35450204,220674,70363,33159,51444,1,0,2,25390,tutor,...,0,3,42000,30799,NaN,88,0,NaN,1,1.0
3,35450295,220674,70363,33110,51395,1,1,1,4859,tutor,...,0,3,4859,30059,NaN,41,0,NaN,2,2.0
4,35450311,220674,70363,33196,51481,1,0,14,19813,tutor,...,3,4,124564,30060,NaN,65,0,0.0,3,3.0


In [ ]:
df.shape

(525534, 30)

In [ ]:
df.columns

Index(['order_id', 'assignment_id', 'user_id', 'assistment_id', 'problem_id',
       'original', 'correct', 'attempt_count', 'ms_first_response',
       'tutor_mode', 'answer_type', 'sequence_id', 'student_class_id',
       'position', 'type', 'base_sequence_id', 'skill_id', 'skill_name',
       'teacher_id', 'school_id', 'hint_count', 'hint_total', 'overlap_time',
       'template_id', 'answer_id', 'answer_text', 'first_action',
       'bottom_hint', 'opportunity', 'opportunity_original'],
      dtype='object')

In [ ]:
df.isnull().sum()

,0
order_id,0
assignment_id,0
user_id,0
assistment_id,0
problem_id,0
original,0
correct,0
attempt_count,0
ms_first_response,0
tutor_mode,0


In [ ]:
df.isnull().sum().sum()

np.int64(1238597)

In [ ]:
# LearnMind ke liye relevant columns
selected_columns = [
    "order_id",
    "user_id",
    "problem_id",
    "correct",
    "attempt_count",
    "ms_first_response",
    "hint_count",
    "skill_id",
    "skill_name",
    "opportunity"
]

learnmind_df = df[selected_columns].copy()

learnmind_df.head()

,order_id,user_id,problem_id,correct,attempt_count,ms_first_response,hint_count,skill_id,skill_name,opportunity
0,33022537,64525,51424,1,1,32454,0,1.0,Box and Whisker,1
1,33022709,64525,51435,1,1,4922,0,1.0,Box and Whisker,2
2,35450204,70363,51444,0,2,25390,0,1.0,Box and Whisker,1
3,35450295,70363,51395,1,1,4859,0,1.0,Box and Whisker,2
4,35450311,70363,51481,0,14,19813,3,1.0,Box and Whisker,3


In [ ]:
learnmind_df.shape

(525534, 10)

In [ ]:
learnmind_df.isnull().sum()

,0
order_id,0
user_id,0
problem_id,0
correct,0
attempt_count,0
ms_first_response,0
hint_count,0
skill_id,66326
skill_name,78690
opportunity,0


In [ ]:
print("Unique skill_id:", learnmind_df["skill_id"].nunique())
print("Unique skill_name:", learnmind_df["skill_name"].nunique())

Unique skill_id: 123
Unique skill_name: 110


In [ ]:
print(
    learnmind_df[
        ["skill_id", "skill_name"]
    ].isnull().all(axis=1).sum()
)

66326


In [ ]:
learnmind_df[
    ["problem_id", "correct", "skill_id", "skill_name"]
].head(20)

,problem_id,correct,skill_id,skill_name
0,51424,1,1.0,Box and Whisker
1,51435,1,1.0,Box and Whisker
2,51444,0,1.0,Box and Whisker
3,51395,1,1.0,Box and Whisker
4,51481,0,1.0,Box and Whisker
5,51457,1,1.0,Box and Whisker
6,51459,1,1.0,Box and Whisker
7,51408,1,1.0,Box and Whisker
8,51453,1,1.0,Box and Whisker
9,51397,1,1.0,Box and Whisker


In [ ]:
# Keep only rows where skill information is available
bkt_df = learnmind_df.dropna(
    subset=["skill_id", "skill_name"]
).copy()

print("Original rows:", len(learnmind_df))
print("Rows after removing missing skills:", len(bkt_df))
print("Rows removed:", len(learnmind_df) - len(bkt_df))

Original rows: 525534
Rows after removing missing skills: 446844
Rows removed: 78690


In [ ]:
print("Missing skill_id:", bkt_df["skill_id"].isnull().sum())
print("Missing skill_name:", bkt_df["skill_name"].isnull().sum())

Missing skill_id: 0
Missing skill_name: 0


In [ ]:
print("Students:", bkt_df["user_id"].nunique())
print("Questions:", bkt_df["problem_id"].nunique())
print("Skills:", bkt_df["skill_id"].nunique())
print("Interactions:", len(bkt_df))

Students: 4151
Questions: 16891
Skills: 111
Interactions: 446844


In [ ]:
bkt_df["skill_name"].value_counts().head(20)

,count
skill_name,
Equation Solving Two or Fewer Steps,24253
Percent Of,22931
Addition and Subtraction Integers,22895
Conversion of Fraction Decimals Percents,20992
Volume Rectangular Prism,19489
Venn Diagram,18275
Volume Cylinder,13734
Multiplication and Division Integers,13159
Area Rectangle,12934


In [ ]:
bkt_df = bkt_df.sort_values(
    ["user_id", "order_id"]
).reset_index(drop=True)

bkt_df.head(10)

,order_id,user_id,problem_id,correct,attempt_count,ms_first_response,hint_count,skill_id,skill_name,opportunity
0,21617623,14,93383,0,1,26271,2,2.0,Circle Graph,1
1,21617623,14,93383,0,1,26271,2,70.0,Percent Of,1
2,21617632,14,93407,1,1,29123,0,2.0,Circle Graph,2
3,21617632,14,93407,1,1,29123,0,70.0,Percent Of,2
4,21617641,14,93400,0,1,13779,2,2.0,Circle Graph,3
5,21617641,14,93400,0,1,13779,2,70.0,Percent Of,3
6,21617650,14,93419,0,1,16901,2,2.0,Circle Graph,4
7,21617650,14,93419,0,1,16901,2,70.0,Percent Of,4
8,21617659,14,93420,0,1,11079,2,2.0,Circle Graph,5
9,21617659,14,93420,0,1,11079,2,70.0,Percent Of,5


In [ ]:
print(bkt_df["correct"].unique())
print(bkt_df["correct"].value_counts())

[0 1]
correct
1    310640
0    136204
Name: count, dtype: int64


In [ ]:
student_id = bkt_df["user_id"].iloc[0]

student_data = bkt_df[
    bkt_df["user_id"] == student_id
][
    ["order_id", "problem_id", "skill_id", "skill_name", "correct"]
]

student_data.head(30)

,order_id,problem_id,skill_id,skill_name,correct
0,21617623,93383,2.0,Circle Graph,0
1,21617623,93383,70.0,Percent Of,0
2,21617632,93407,2.0,Circle Graph,1
3,21617632,93407,70.0,Percent Of,1
4,21617641,93400,2.0,Circle Graph,0
5,21617641,93400,70.0,Percent Of,0
6,21617650,93419,2.0,Circle Graph,0
7,21617650,93419,70.0,Percent Of,0
8,21617659,93420,2.0,Circle Graph,0
9,21617659,93420,70.0,Percent Of,0


In [ ]:
problem_skill_count = (
    bkt_df.groupby("problem_id")["skill_id"]
    .nunique()
)

print(problem_skill_count.value_counts().head())

skill_id
1    13905
2     2664
3      297
4       25
Name: count, dtype: int64


In [ ]:
print(
    "Problems with multiple skills:",
    (problem_skill_count > 1).sum()
)

Problems with multiple skills: 2986


In [ ]:
problem_skill_count = (
    bkt_df.groupby("problem_id")["skill_id"]
    .nunique()
)

single_skill_problems = (problem_skill_count == 1).sum()
multi_skill_problems = (problem_skill_count > 1).sum()

print("Single-skill problems:", single_skill_problems)
print("Multi-skill problems:", multi_skill_problems)

Single-skill problems: 13905
Multi-skill problems: 2986


In [ ]:
multi_skill_problem_ids = problem_skill_count[
    problem_skill_count > 1
].index

multi_skill_interactions = bkt_df[
    bkt_df["problem_id"].isin(multi_skill_problem_ids)
]

print("Total interactions:", len(bkt_df))
print("Multi-skill interactions:", len(multi_skill_interactions))
print(
    "Percentage:",
    round(len(multi_skill_interactions) / len(bkt_df) * 100, 2),
    "%"
)

Total interactions: 446844
Multi-skill interactions: 97986
Percentage: 21.93 %


In [ ]:
bkt_df.groupby("problem_id")["skill_id"].nunique().value_counts().sort_index()

,count
skill_id,
1,13905
2,2664
3,297
4,25


In [ ]:
# Identify single-skill problems
problem_skill_count = (
    bkt_df.groupby("problem_id")["skill_id"]
    .nunique()
)

single_skill_problem_ids = problem_skill_count[
    problem_skill_count == 1
].index

# Keep only single-skill problems
bkt_single = bkt_df[
    bkt_df["problem_id"].isin(single_skill_problem_ids)
].copy()

print("Original interactions:", len(bkt_df))
print("BKT interactions:", len(bkt_single))
print("Students:", bkt_single["user_id"].nunique())
print("Questions:", bkt_single["problem_id"].nunique())
print("Skills:", bkt_single["skill_id"].nunique())

Original interactions: 446844
BKT interactions: 348858
Students: 3991
Questions: 13905
Skills: 97


In [ ]:
bkt_single[
    ["user_id", "problem_id", "skill_id", "skill_name", "correct"]
].head(20)

,user_id,problem_id,skill_id,skill_name,correct
28,14,91522,13.0,Median,0
29,14,92390,15.0,Range,1
30,14,92379,15.0,Range,1
31,14,91144,13.0,Median,0
32,14,92361,15.0,Range,1
33,14,91124,13.0,Median,0
34,14,91127,13.0,Median,0
49,21825,90543,279.0,Multiplication and Division Integers,1
50,21825,90594,279.0,Multiplication and Division Integers,1
51,21825,90425,279.0,Multiplication and Division Integers,1


In [ ]:
bkt_single["skill_id"] = bkt_single["skill_id"].astype(int)

In [ ]:
bkt_single["skill_id"].unique()[:20]

array([ 13,  15, 279, 311, 312, 278,  86, 340, 277,  24,  54,  67, 280,
        58,  51,  49, 309, 368, 301,  32])

In [ ]:
student_id = bkt_single["user_id"].iloc[0]

student_sequence = bkt_single[
    bkt_single["user_id"] == student_id
].sort_values("order_id")

student_sequence[
    ["order_id", "problem_id", "skill_id", "skill_name", "correct"]
].head(30)

,order_id,problem_id,skill_id,skill_name,correct
28,21617893,91522,13,Median,0
29,21617915,92390,15,Range,1
30,21617924,92379,15,Range,1
31,21617931,91144,13,Median,0
32,21617942,92361,15,Range,1
33,21617953,91124,13,Median,0
34,21617962,91127,13,Median,0


In [ ]:
baseline = (
    bkt_single
    .groupby(["user_id", "skill_id", "skill_name"])
    .agg(
        attempts=("correct", "count"),
        correct=("correct", "sum"),
        accuracy=("correct", "mean")
    )
    .reset_index()
)

baseline["mastery"] = baseline["accuracy"]

baseline.head(10)

,user_id,skill_id,skill_name,attempts,correct,accuracy,mastery
0,14,13,Median,4,0,0.00,0.00
1,14,15,Range,3,3,1.00,1.00
2,21825,279,Multiplication and Division Integers,10,10,1.00,1.00
3,51950,311,Equation Solving Two or Fewer Steps,4,3,0.75,0.75
4,51950,312,Equation Solving More Than Two Steps,2,2,1.00,1.00
5,52613,86,Exponents,1,1,1.00,1.00
6,52613,278,Addition and Subtraction Positive Decimals,1,0,0.00,0.00
7,52613,340,Distributive Property,3,3,1.00,1.00
8,53167,8,Scatter Plot,5,3,0.60,0.60
9,53167,12,Mean,1,0,0.00,0.00


In [ ]:
print("Student-Skill pairs:", len(baseline))
print("Average attempts per skill:", baseline["attempts"].mean())
print("Average baseline mastery:", baseline["mastery"].mean())

Student-Skill pairs: 30244
Average attempts per skill: 11.53478375876207
Average baseline mastery: 0.731632951511239


In [ ]:
baseline["mastery"].describe()

,mastery
count,30244.000000
mean,0.731633
std,0.320340
min,0.000000
25%,0.571429
50%,0.833333
75%,1.000000
max,1.000000


In [ ]:
def bkt_update(p_know, correct,
               p_transit=0.10,
               p_guess=0.20,
               p_slip=0.10):

    # Probability of observing the response
    if correct == 1:
        p_correct = (
            p_know * (1 - p_slip)
            + (1 - p_know) * p_guess
        )

        # Bayesian update after correct answer
        p_know_given_response = (
            p_know * (1 - p_slip)
        ) / p_correct

    else:
        p_incorrect = (
            p_know * p_slip
            + (1 - p_know) * (1 - p_guess)
        )

        # Bayesian update after incorrect answer
        p_know_given_response = (
            p_know * p_slip
        ) / p_incorrect

    # Learning transition
    p_know_after_learning = (
        p_know_given_response
        + (1 - p_know_given_response) * p_transit
    )

    return p_know_after_learning

In [ ]:
p = 0.20

responses = [1, 1, 0, 1, 1]

for i, response in enumerate(responses, 1):
    p = bkt_update(p, response)
    print(f"After attempt {i}, response={response}, mastery={p:.4f}")

After attempt 1, response=1, mastery=0.5765
After attempt 2, response=1, mastery=0.8737
After attempt 3, response=0, mastery=0.5173
After attempt 4, response=1, mastery=0.8454
After attempt 5, response=1, mastery=0.9649


In [ ]:
def apply_bkt(df,
              p_l0=0.20,
              p_transit=0.10,
              p_guess=0.20,
              p_slip=0.10):

    df = df.sort_values(["user_id", "order_id"]).copy()

    bkt_mastery = []
    bkt_prediction = []

    current_student = None
    current_skill = None
    p_know = p_l0

    for _, row in df.iterrows():

        student = row["user_id"]
        skill = row["skill_id"]
        correct = row["correct"]

        # New student-skill pair
        if student != current_student or skill != current_skill:
            current_student = student
            current_skill = skill
            p_know = p_l0

        # Prediction BEFORE observing the answer
        p_correct = (
            p_know * (1 - p_slip)
            + (1 - p_know) * p_guess
        )

        bkt_prediction.append(p_correct)

        # Update knowledge AFTER observing the answer
        p_know = bkt_update(
            p_know,
            correct,
            p_transit=p_transit,
            p_guess=p_guess,
            p_slip=p_slip
        )

        bkt_mastery.append(p_know)

    df["bkt_prediction"] = bkt_prediction
    df["bkt_mastery"] = bkt_mastery

    return df

In [ ]:
bkt_result = apply_bkt(bkt_single)

print("Rows:", len(bkt_result))
print("BKT mastery mean:", bkt_result["bkt_mastery"].mean())
print("BKT mastery min:", bkt_result["bkt_mastery"].min())
print("BKT mastery max:", bkt_result["bkt_mastery"].max())

Rows: 348858
BKT mastery mean: 0.7120459902608658
BKT mastery min: 0.11428571428571428
BKT mastery max: 1.0


In [ ]:
bkt_result[
    ["user_id", "problem_id", "skill_id",
     "skill_name", "correct",
     "bkt_prediction", "bkt_mastery"]
].head(20)

,user_id,problem_id,skill_id,skill_name,correct,bkt_prediction,bkt_mastery
28,14,91522,13,Median,0,0.340000,0.127273
29,14,92390,15,Range,1,0.340000,0.576471
30,14,92379,15,Range,1,0.603529,0.873684
31,14,91144,13,Median,0,0.340000,0.127273
32,14,92361,15,Range,1,0.340000,0.576471
33,14,91124,13,Median,0,0.340000,0.127273
34,14,91127,13,Median,0,0.289091,0.116113
49,21825,90543,279,Multiplication and Division Integers,1,0.340000,0.576471
50,21825,90594,279,Multiplication and Division Integers,1,0.603529,0.873684
51,21825,90425,279,Multiplication and Division Integers,1,0.811579,0.971984


In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

y_true = bkt_result["correct"]
y_pred_prob = bkt_result["bkt_prediction"]

# Convert probability to binary prediction
y_pred = (y_pred_prob >= 0.5).astype(int)

accuracy = accuracy_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_pred_prob)
loss = log_loss(y_true, y_pred_prob)

print(f"BKT Accuracy : {accuracy:.4f}")
print(f"BKT ROC-AUC  : {auc:.4f}")
print(f"BKT Log Loss : {loss:.4f}")

BKT Accuracy : 0.7303
BKT ROC-AUC  : 0.7756
BKT Log Loss : 0.5385


In [ ]:
baseline_prob = y_true.mean()

baseline_pred = (baseline_prob >= 0.5).astype(int)

baseline_accuracy = accuracy_score(
    y_true,
    [baseline_pred] * len(y_true)
)

baseline_auc = roc_auc_score(
    y_true,
    [baseline_prob] * len(y_true)
)

baseline_loss = log_loss(
    y_true,
    [baseline_prob] * len(y_true)
)

print(f"Baseline Accuracy : {baseline_accuracy:.4f}")
print(f"Baseline ROC-AUC  : {baseline_auc:.4f}")
print(f"Baseline Log Loss : {baseline_loss:.4f}")

Baseline Accuracy : 0.7095
Baseline ROC-AUC  : 0.5000
Baseline Log Loss : 0.6026


In [ ]:
# Use first 50,000 interactions for parameter search
opt_df = bkt_single.sort_values(
    ["user_id", "order_id"]
).head(50000).copy()

print("Optimization dataset:", len(opt_df))
print("Students:", opt_df["user_id"].nunique())
print("Skills:", opt_df["skill_id"].nunique())

Optimization dataset: 50000
Students: 444
Skills: 90


In [ ]:
from itertools import product
from sklearn.metrics import log_loss

parameter_grid = {
    "p_l0": [0.1, 0.2, 0.3],
    "p_transit": [0.05, 0.10, 0.20],
    "p_guess": [0.10, 0.20, 0.30],
    "p_slip": [0.05, 0.10, 0.20]
}

best_loss = float("inf")
best_params = None

for p_l0, p_transit, p_guess, p_slip in product(
    parameter_grid["p_l0"],
    parameter_grid["p_transit"],
    parameter_grid["p_guess"],
    parameter_grid["p_slip"]
):

    temp = apply_bkt(
        opt_df,
        p_l0=p_l0,
        p_transit=p_transit,
        p_guess=p_guess,
        p_slip=p_slip
    )

    loss = log_loss(
        temp["correct"],
        temp["bkt_prediction"]
    )

    if loss < best_loss:
        best_loss = loss
        best_params = {
            "p_l0": p_l0,
            "p_transit": p_transit,
            "p_guess": p_guess,
            "p_slip": p_slip
        }

print("Best parameters:")
print(best_params)
print(f"Best Log Loss: {best_loss:.4f}")

Best parameters:
{'p_l0': 0.3, 'p_transit': 0.2, 'p_guess': 0.3, 'p_slip': 0.1}
Best Log Loss: 0.5228


In [ ]:
optimized_bkt = apply_bkt(
    bkt_single,
    p_l0=0.30,
    p_transit=0.20,
    p_guess=0.30,
    p_slip=0.10
)

print("Rows:", len(optimized_bkt))
print("Mean BKT mastery:", optimized_bkt["bkt_mastery"].mean())
print("Mean prediction:", optimized_bkt["bkt_prediction"].mean())

Rows: 348858
Mean BKT mastery: 0.7575201773029598
Mean prediction: 0.7164234155872085


In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, log_loss

y_true = optimized_bkt["correct"]
y_prob = optimized_bkt["bkt_prediction"]
y_pred = (y_prob >= 0.5).astype(int)

print(f"Optimized BKT Accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"Optimized BKT ROC-AUC  : {roc_auc_score(y_true, y_prob):.4f}")
print(f"Optimized BKT Log Loss : {log_loss(y_true, y_prob):.4f}")

Optimized BKT Accuracy : 0.7378
Optimized BKT ROC-AUC  : 0.7760
Optimized BKT Log Loss : 0.5174


In [ ]:
data = bkt_single.sort_values(
    ["user_id", "order_id"]
).copy()

# Position of each interaction within each student's sequence
data["interaction_number"] = (
    data.groupby("user_id").cumcount()
)

data["total_interactions"] = (
    data.groupby("user_id")["interaction_number"]
    .transform("max") + 1
)

data["train_cutoff"] = (
    data["total_interactions"] * 0.8
).astype(int)

train_df = data[
    data["interaction_number"] < data["train_cutoff"]
].copy()

test_df = data[
    data["interaction_number"] >= data["train_cutoff"]
].copy()

print("Total:", len(data))
print("Train:", len(train_df))
print("Test:", len(test_df))
print("Train %:", len(train_df) / len(data))
print("Test %:", len(test_df) / len(data))

Total: 348858
Train: 277447
Test: 71411
Train %: 0.7953006667469286
Test %: 0.20469933325307146


In [ ]:
print("Train students:", train_df["user_id"].nunique())
print("Test students:", test_df["user_id"].nunique())

print("\nTrain date/order range:")
print(train_df["order_id"].min(), "→", train_df["order_id"].max())

print("\nTest date/order range:")
print(test_df["order_id"].min(), "→", test_df["order_id"].max())

Train students: 3833
Test students: 3991

Train date/order range:
20224085 → 38310005

Test date/order range:
20277622 → 38310197


In [ ]:
def initialize_bkt_states(df, p_l0=0.30):
    states = {}

    for _, row in df.iterrows():
        key = (row["user_id"], row["skill_id"])

        if key not in states:
            states[key] = p_l0

        states[key] = bkt_update(
            states[key],
            row["correct"],
            p_transit=0.20,
            p_guess=0.30,
            p_slip=0.10
        )

    return states

In [ ]:
bkt_states = initialize_bkt_states(train_df)

print("Student-skill states:", len(bkt_states))

Student-skill states: 25638


In [ ]:
def evaluate_online_bkt(test_df, initial_states,
                        p_l0=0.30,
                        p_transit=0.20,
                        p_guess=0.30,
                        p_slip=0.10):

    states = initial_states.copy()

    predictions = []
    actuals = []

    for _, row in test_df.sort_values(
        ["user_id", "order_id"]
    ).iterrows():

        key = (row["user_id"], row["skill_id"])

        # If this student-skill pair was never seen during training
        if key not in states:
            states[key] = p_l0

        p_know = states[key]

        # Predict BEFORE seeing answer
        p_correct = (
            p_know * (1 - p_slip)
            + (1 - p_know) * p_guess
        )

        predictions.append(p_correct)
        actuals.append(row["correct"])

        # Now observe answer and update state
        states[key] = bkt_update(
            p_know,
            row["correct"],
            p_transit=p_transit,
            p_guess=p_guess,
            p_slip=p_slip
        )

    return actuals, predictions, states

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    log_loss
)

y_test, test_prob, final_states = evaluate_online_bkt(
    test_df,
    bkt_states
)

test_pred = [
    1 if p >= 0.5 else 0
    for p in test_prob
]

print(f"Test Accuracy : {accuracy_score(y_test, test_pred):.4f}")
print(f"Test ROC-AUC  : {roc_auc_score(y_test, test_prob):.4f}")
print(f"Test Log Loss : {log_loss(y_test, test_prob):.4f}")

Test Accuracy : 0.7689
Test ROC-AUC  : 0.7543
Test Log Loss : 0.5149


In [ ]:
mastery_table = (
    pd.DataFrame([
        {
            "user_id": user_id,
            "skill_id": skill_id,
            "bkt_mastery": mastery
        }
        for (user_id, skill_id), mastery in final_states.items()
    ])
)

print("Rows:", len(mastery_table))
print("Students:", mastery_table["user_id"].nunique())
print("Skills:", mastery_table["skill_id"].nunique())

mastery_table.head(10)

Rows: 30244
Students: 3991
Skills: 97


,user_id,skill_id,bkt_mastery
0,14,13,0.233408
1,14,15,0.964669
2,21825,279,0.999996
3,51950,311,0.956375
4,52613,278,0.246154
5,52613,86,0.650000
6,52613,340,0.964669
7,53167,278,0.968875
8,53167,277,0.999742
9,53167,279,0.999995


In [ ]:
skill_lookup = (
    bkt_single[["skill_id", "skill_name"]]
    .drop_duplicates("skill_id")
)

mastery_table = mastery_table.merge(
    skill_lookup,
    on="skill_id",
    how="left"
)

mastery_table = mastery_table[
    ["user_id", "skill_id", "skill_name", "bkt_mastery"]
]

mastery_table.head(10)

,user_id,skill_id,skill_name,bkt_mastery
0,14,13,Median,0.233408
1,14,15,Range,0.964669
2,21825,279,Multiplication and Division Integers,0.999996
3,51950,311,Equation Solving Two or Fewer Steps,0.956375
4,52613,278,Addition and Subtraction Positive Decimals,0.246154
5,52613,86,Exponents,0.650000
6,52613,340,Distributive Property,0.964669
7,53167,278,Addition and Subtraction Positive Decimals,0.968875
8,53167,277,Addition and Subtraction Integers,0.999742
9,53167,279,Multiplication and Division Integers,0.999995


In [ ]:
print(mastery_table["bkt_mastery"].describe())

count    30244.000000
mean         0.788973
std          0.253763
min          0.233333
25%          0.650000
50%          0.887924
75%          0.997410
max          1.000000
Name: bkt_mastery, dtype: float64


**Weak Concept Detection**

In [ ]:
def mastery_level(x):
    if x < 0.50:
        return "Weak"
    elif x < 0.70:
        return "Developing"
    else:
        return "Strong"

mastery_table["mastery_level"] = (
    mastery_table["bkt_mastery"]
    .apply(mastery_level)
)

print(
    mastery_table["mastery_level"]
    .value_counts()
)

mastery_level
Strong        18966
Developing     7120
Weak           4158
Name: count, dtype: int64


In [ ]:
mastery_distribution = (
    mastery_table["mastery_level"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(mastery_distribution)

mastery_level
Strong        62.71
Developing    23.54
Weak          13.75
Name: proportion, dtype: float64


In [ ]:
weak_concepts = (
    mastery_table[
        mastery_table["mastery_level"] == "Weak"
    ]
    .sort_values("bkt_mastery")
)

print("Weak student-skill pairs:", len(weak_concepts))

weak_concepts.head(20)

Weak student-skill pairs: 4158


,user_id,skill_id,skill_name,bkt_mastery,mastery_level
12408,80228,81,Unit Rate,0.233333,Weak
6327,78927,303,Volume Cylinder,0.233333,Weak
19002,88127,278,Addition and Subtraction Positive Decimals,0.233333,Weak
5231,78875,303,Volume Cylinder,0.233333,Weak
21353,92103,86,Exponents,0.233333,Weak
9712,79333,297,Area Trapezoid,0.233333,Weak
3844,78493,303,Volume Cylinder,0.233333,Weak
19115,88153,278,Addition and Subtraction Positive Decimals,0.233333,Weak
16611,85043,280,Addition and Subtraction Fractions,0.233333,Weak
9530,79220,77,Finding Percents,0.233333,Weak


**Smarter Weakness Score**

In [ ]:
# Get recent performance for each student-skill pair

recent_performance = (
    bkt_single
    .sort_values(["user_id", "skill_id", "order_id"])
    .groupby(["user_id", "skill_id"])
    .tail(5)
    .groupby(["user_id", "skill_id"])["correct"]
    .mean()
    .reset_index(name="recent_accuracy")
)

# Merge with BKT mastery
weakness_table = mastery_table.merge(
    recent_performance,
    on=["user_id", "skill_id"],
    how="left"
)

# Get number of attempts
attempt_counts = (
    bkt_single
    .groupby(["user_id", "skill_id"])
    .size()
    .reset_index(name="attempts")
)

weakness_table = weakness_table.merge(
    attempt_counts,
    on=["user_id", "skill_id"],
    how="left"
)

weakness_table.head()

,user_id,skill_id,skill_name,bkt_mastery,mastery_level,recent_accuracy,attempts
0,14,13,Median,0.233408,Weak,0.00,4
1,14,15,Range,0.964669,Strong,1.00,3
2,21825,279,Multiplication and Division Integers,0.999996,Strong,1.00,10
3,51950,311,Equation Solving Two or Fewer Steps,0.956375,Strong,0.75,4
4,52613,278,Addition and Subtraction Positive Decimals,0.246154,Weak,0.00,1


**Weakness Score**

In [ ]:
# Recalculate weakness score with evidence confidence

weakness_table["mastery_weakness"] = (
    1 - weakness_table["bkt_mastery"]
)

weakness_table["recent_weakness"] = (
    1 - weakness_table["recent_accuracy"]
)

# Confidence increases as the student has more attempts
weakness_table["evidence_confidence"] = (
    weakness_table["attempts"] /
    (weakness_table["attempts"] + 5)
)

# Raw weakness based on mastery + recent performance
weakness_table["raw_weakness"] = (
    0.70 * weakness_table["mastery_weakness"]
    + 0.30 * weakness_table["recent_weakness"]
)

# Confidence-adjusted weakness
weakness_table["weakness_score"] = (
    weakness_table["raw_weakness"]
    * weakness_table["evidence_confidence"]
)

# Sort highest priority first
weakness_table = weakness_table.sort_values(
    "weakness_score",
    ascending=False
)

print(
    weakness_table[
        [
            "user_id",
            "skill_id",
            "skill_name",
            "bkt_mastery",
            "recent_accuracy",
            "attempts",
            "raw_weakness",
            "evidence_confidence",
            "weakness_score"
        ]
    ].head(20)
)

       user_id  skill_id                                skill_name  \
4012     78521       303                           Volume Cylinder   
6327     78927       303                           Volume Cylinder   
4472     78591       297                            Area Trapezoid   
4027     78523       303                           Volume Cylinder   
5231     78875       303                           Volume Cylinder   
4021     78522       297                            Area Trapezoid   
4234     78561       297                            Area Trapezoid   
14840    83656       277         Addition and Subtraction Integers   
3437     78341       311       Equation Solving Two or Fewer Steps   
13615    82080        47  Conversion of Fraction Decimals Percents   
14393    83291       277         Addition and Subtraction Integers   
26470    78574        70                                Percent Of   
17619    86318       325          Write Linear Equation from Graph   
4299     78571      

**Recommendation Engine**

In [ ]:
def get_recommendation(row):
    mastery = row["bkt_mastery"]
    recent = row["recent_accuracy"]
    attempts = row["attempts"]

    if mastery < 0.35 and recent <= 0.20:
        priority = "Critical"
        action = "Relearn concept from basics"

    elif mastery < 0.50 and recent <= 0.40:
        priority = "High"
        action = "Review concept and practice more questions"

    elif mastery < 0.70:
        priority = "Medium"
        action = "Practice targeted questions"

    else:
        priority = "Low"
        action = "Continue normal practice"

    return pd.Series({
        "priority": priority,
        "recommended_action": action
    })


recommendations = weakness_table.apply(
    get_recommendation,
    axis=1
)

weakness_table = pd.concat(
    [weakness_table, recommendations],
    axis=1
)

print(
    weakness_table[
        [
            "user_id",
            "skill_id",
            "skill_name",
            "bkt_mastery",
            "recent_accuracy",
            "attempts",
            "weakness_score",
            "priority",
            "recommended_action"
        ]
    ].head(20)
)

       user_id  skill_id                                skill_name  \
4012     78521       303                           Volume Cylinder   
6327     78927       303                           Volume Cylinder   
4472     78591       297                            Area Trapezoid   
4027     78523       303                           Volume Cylinder   
5231     78875       303                           Volume Cylinder   
4021     78522       297                            Area Trapezoid   
4234     78561       297                            Area Trapezoid   
14840    83656       277         Addition and Subtraction Integers   
3437     78341       311       Equation Solving Two or Fewer Steps   
13615    82080        47  Conversion of Fraction Decimals Percents   
14393    83291       277         Addition and Subtraction Integers   
26470    78574        70                                Percent Of   
17619    86318       325          Write Linear Equation from Graph   
4299     78571      

**Find Practice Questions for Weak Skills**

In [ ]:
# Build skill -> question mapping
# Only use single-skill questions from our BKT dataset

question_bank = (
    bkt_single[
        [
            "problem_id",
            "skill_id",
            "skill_name",
            "correct",
            "attempt_count",
            "hint_count"
        ]
    ]
    .drop_duplicates(
        subset=["problem_id", "skill_id"]
    )
)

print("Unique practice questions:", len(question_bank))
print("Unique skills:", question_bank["skill_id"].nunique())

# Example: questions for the most common weak skill
top_weak_skill = weakness_table.iloc[0]["skill_id"]

skill_questions = question_bank[
    question_bank["skill_id"] == top_weak_skill
]

print("\nTop weak skill:")
print(weakness_table.iloc[0][
    ["user_id", "skill_id", "skill_name", "bkt_mastery",
     "recent_accuracy", "attempts", "weakness_score"]
])

print("\nAvailable questions for this skill:",
      len(skill_questions))

skill_questions.head(10)

Unique practice questions: 13905
Unique skills: 97

Top weak skill:
user_id                      78521
skill_id                       303
skill_name         Volume Cylinder
bkt_mastery               0.233333
recent_accuracy                0.0
attempts                       279
weakness_score            0.821937
Name: 4012, dtype: object

Available questions for this skill: 70


,problem_id,skill_id,skill_name,correct,attempt_count,hint_count
1113,75749,303,Volume Cylinder,0,5,5
1114,75750,303,Volume Cylinder,1,1,0
1115,75737,303,Volume Cylinder,1,1,0
1116,75732,303,Volume Cylinder,1,1,0
1117,89779,303,Volume Cylinder,1,1,0
11891,89766,303,Volume Cylinder,0,1,3
13432,75746,303,Volume Cylinder,0,3,5
19064,75741,303,Volume Cylinder,0,1,0
20873,75742,303,Volume Cylinder,0,4,5
20874,75738,303,Volume Cylinder,0,3,0


**Estimate Question Difficulty**

In [ ]:
# Estimate question difficulty from historical responses

question_difficulty = (
    bkt_single
    .groupby(["problem_id", "skill_id", "skill_name"])
    .agg(
        attempts=("correct", "count"),
        success_rate=("correct", "mean")
    )
    .reset_index()
)

# Convert success rate into difficulty
question_difficulty["difficulty"] = (
    1 - question_difficulty["success_rate"]
)

# Sort from easiest to hardest
question_difficulty = question_difficulty.sort_values(
    ["skill_id", "difficulty"]
)

print("Questions:", len(question_difficulty))
print("\nDifficulty statistics:")
print(question_difficulty["difficulty"].describe())

# Show questions for the student's weakest skill
skill_questions = question_difficulty[
    question_difficulty["skill_id"] == top_weak_skill
]

print("\nQuestions for:", weakness_table.iloc[0]["skill_name"])

print(
    skill_questions[
        [
            "problem_id",
            "skill_id",
            "skill_name",
            "attempts",
            "success_rate",
            "difficulty"
        ]
    ].head(20)
)

Questions: 13905

Difficulty statistics:
count    13905.000000
mean         0.317683
std          0.280846
min          0.000000
25%          0.076923
50%          0.272727
75%          0.500000
max          1.000000
Name: difficulty, dtype: float64

Questions for: Volume Cylinder
      problem_id  skill_id       skill_name  attempts  success_rate  \
8153       89770       303  Volume Cylinder       357      0.997199   
8167       89784       303  Volume Cylinder       214      0.967290   
8149       89766       303  Volume Cylinder       244      0.897541   
9252       91644       303  Volume Cylinder       524      0.879771   
8147       89764       303  Volume Cylinder       288      0.850694   
8157       89774       303  Volume Cylinder       247      0.817814   
3797       75725       303  Volume Cylinder        32      0.812500   
9246       91638       303  Volume Cylinder       338      0.804734   
8172       89789       303  Volume Cylinder       442      0.789593   
8154    

**Adaptive Question Selection**

In [ ]:
def recommend_questions(
    user_id,
    skill_id,
    mastery,
    question_df,
    interaction_df,
    n_questions=5
):
    # Questions already attempted by this student
    attempted = set(
        interaction_df[
            interaction_df["user_id"] == user_id
        ]["problem_id"]
    )

    # Only questions from the target skill
    candidates = question_df[
        question_df["skill_id"] == skill_id
    ].copy()

    # Remove already attempted questions
    candidates = candidates[
        ~candidates["problem_id"].isin(attempted)
    ].copy()

    if len(candidates) == 0:
        return candidates

    # Prefer questions whose difficulty is close
    # to the student's current mastery
    candidates["difficulty_gap"] = (
        candidates["difficulty"] - mastery
    ).abs()

    # Closest difficulty first
    candidates = candidates.sort_values(
        "difficulty_gap"
    )

    return candidates.head(n_questions)

In [ ]:
top_row = weakness_table.iloc[0]

recommended = recommend_questions(
    user_id=top_row["user_id"],
    skill_id=top_row["skill_id"],
    mastery=top_row["bkt_mastery"],
    question_df=question_difficulty,
    interaction_df=bkt_single,
    n_questions=5
)

print(
    recommended[
        [
            "problem_id",
            "skill_id",
            "skill_name",
            "attempts",
            "success_rate",
            "difficulty",
            "difficulty_gap"
        ]
    ]
)

      problem_id  skill_id       skill_name  attempts  success_rate  \
8154       89771       303  Volume Cylinder       269      0.762082   
3814       75742       303  Volume Cylinder        29      0.758621   
8156       89773       303  Volume Cylinder       266      0.748120   
8172       89789       303  Volume Cylinder       442      0.789593   
3812       75740       303  Volume Cylinder        27      0.740741   

      difficulty  difficulty_gap  
8154    0.237918        0.004585  
3814    0.241379        0.008046  
8156    0.251880        0.018546  
8172    0.210407        0.022926  
3812    0.259259        0.025926  


**Progressive Adaptive Test**

In [ ]:
def adaptive_test(
    user_id,
    skill_id,
    mastery,
    question_df,
    interaction_df,
    n_questions=5
):
    # Questions already attempted
    attempted = set(
        interaction_df[
            interaction_df["user_id"] == user_id
        ]["problem_id"]
    )

    # Target skill
    candidates = question_df[
        question_df["skill_id"] == skill_id
    ].copy()

    # Remove previously attempted questions
    candidates = candidates[
        ~candidates["problem_id"].isin(attempted)
    ].copy()

    if len(candidates) == 0:
        return candidates

    # Difficulty progression
    target_difficulties = [
        mastery - 0.10,
        mastery - 0.03,
        mastery,
        mastery + 0.07,
        mastery + 0.15
    ]

    selected = []

    for target in target_difficulties:

        remaining = candidates[
            ~candidates["problem_id"].isin(
                [q["problem_id"] for q in selected]
            )
        ].copy()

        if len(remaining) == 0:
            break

        remaining["difficulty_gap"] = (
            remaining["difficulty"] - target
        ).abs()

        best = remaining.loc[
            remaining["difficulty_gap"].idxmin()
        ]

        selected.append(
            best.to_dict()
        )

    return pd.DataFrame(selected)

In [ ]:
adaptive_recommendations = adaptive_test(
    user_id=top_row["user_id"],
    skill_id=top_row["skill_id"],
    mastery=top_row["bkt_mastery"],
    question_df=question_difficulty,
    interaction_df=bkt_single,
    n_questions=5
)

print(
    adaptive_recommendations[
        [
            "problem_id",
            "skill_name",
            "difficulty",
            "difficulty_gap"
        ]
    ]
)

   problem_id       skill_name  difficulty  difficulty_gap
0       91644  Volume Cylinder    0.120229        0.013104
1       89789  Volume Cylinder    0.210407        0.007074
2       89771  Volume Cylinder    0.237918        0.004585
3       75749  Volume Cylinder    0.297297        0.006036
4       91642  Volume Cylinder    0.384401        0.001068


**Simulate an Adaptive Learning Session**

In [ ]:
def simulate_adaptive_session(
    user_id,
    skill_id,
    initial_mastery,
    question_df,
    interaction_df,
    n_questions=5
):
    mastery = initial_mastery
    session = []

    attempted = set(
        interaction_df[
            interaction_df["user_id"] == user_id
        ]["problem_id"]
    )

    candidates = question_df[
        question_df["skill_id"] == skill_id
    ].copy()

    candidates = candidates[
        ~candidates["problem_id"].isin(attempted)
    ].copy()

    for i in range(n_questions):

        if len(candidates) == 0:
            break

        # Select question closest to current mastery
        candidates["difficulty_gap"] = (
            candidates["difficulty"] - mastery
        ).abs()

        question = candidates.loc[
            candidates["difficulty_gap"].idxmin()
        ]

        # Historical success rate acts as simulated probability
        simulated_correct = int(
            np.random.rand() < question["success_rate"]
        )

        mastery_before = mastery

        # BKT update
        mastery = bkt_update(
            mastery,
            simulated_correct,
            p_transit=0.20,
            p_guess=0.30,
            p_slip=0.10
        )

        session.append({
            "question_number": i + 1,
            "problem_id": question["problem_id"],
            "difficulty": question["difficulty"],
            "simulated_correct": simulated_correct,
            "mastery_before": mastery_before,
            "mastery_after": mastery
        })

        # Remove selected question
        candidates = candidates[
            candidates["problem_id"] != question["problem_id"]
        ]

    return pd.DataFrame(session)

In [ ]:
np.random.seed(42)

adaptive_session = simulate_adaptive_session(
    user_id=top_row["user_id"],
    skill_id=top_row["skill_id"],
    initial_mastery=top_row["bkt_mastery"],
    question_df=question_difficulty,
    interaction_df=bkt_single,
    n_questions=5
)

adaptive_session

,question_number,problem_id,difficulty,simulated_correct,mastery_before,mastery_after
0,1,89771,0.237918,1,0.233333,0.581818
1,2,75738,0.562500,0,0.581818,0.332642
2,3,89781,0.327273,0,0.332642,0.253179
3,4,89773,0.251880,1,0.253179,0.603376
4,5,75737,0.606061,1,0.603376,0.856215


In [ ]:
def adaptive_difficulty_target(
    mastery,
    previous_correct=None
):
    # First question: slightly below current mastery
    if previous_correct is None:
        return max(0.05, mastery - 0.10)

    # Correct → challenge the student
    if previous_correct == 1:
        return min(0.95, mastery + 0.10)

    # Incorrect → step down and rebuild confidence
    return max(0.05, mastery - 0.10)

In [ ]:
mastery = top_row["bkt_mastery"]

results = []

previous_correct = None

for i in range(5):

    target = adaptive_difficulty_target(
        mastery,
        previous_correct
    )

    results.append({
        "question_number": i + 1,
        "mastery": mastery,
        "previous_correct": previous_correct,
        "target_difficulty": target
    })

    # Simulate a response for testing
    previous_correct = 1 if i % 2 == 0 else 0

    mastery = bkt_update(
        mastery,
        previous_correct,
        p_transit=0.20,
        p_guess=0.30,
        p_slip=0.10
    )

pd.DataFrame(results)

,question_number,mastery,previous_correct,target_difficulty
0,1,0.233333,NaN,0.133333
1,2,0.581818,1.0,0.681818
2,3,0.332642,0.0,0.232642
3,4,0.679403,1.0,0.779403
4,5,0.385909,0.0,0.285909


In [ ]:
def select_adaptive_question(
    user_id,
    skill_id,
    mastery,
    previous_correct,
    question_df,
    interaction_df
):
    target = adaptive_difficulty_target(
        mastery,
        previous_correct
    )

    # Questions for this skill
    candidates = question_df[
        question_df["skill_id"] == skill_id
    ].copy()

    # Remove questions already attempted by this student
    attempted = set(
        interaction_df[
            interaction_df["user_id"] == user_id
        ]["problem_id"]
    )

    candidates = candidates[
        ~candidates["problem_id"].isin(attempted)
    ].copy()

    if len(candidates) == 0:
        return None

    # Find question closest to target difficulty
    candidates["difficulty_gap"] = (
        candidates["difficulty"] - target
    ).abs()

    question = candidates.loc[
        candidates["difficulty_gap"].idxmin()
    ]

    return {
        "problem_id": question["problem_id"],
        "skill_id": question["skill_id"],
        "skill_name": question["skill_name"],
        "difficulty": question["difficulty"],
        "target_difficulty": target,
        "difficulty_gap": question["difficulty_gap"]
    }

In [ ]:
top_row = weakness_table.iloc[0]

selected_question = select_adaptive_question(
    user_id=top_row["user_id"],
    skill_id=top_row["skill_id"],
    mastery=top_row["bkt_mastery"],
    previous_correct=None,
    question_df=question_difficulty,
    interaction_df=bkt_single
)

selected_question

{'problem_id': np.int64(91644),
 'skill_id': np.int64(303),
 'skill_name': 'Volume Cylinder',
 'difficulty': np.float64(0.12022900763358779),
 'target_difficulty': np.float64(0.13333333333333333),
 'difficulty_gap': np.float64(0.01310432569974554)}

**Full Adaptive Test Loop**

In [ ]:
def run_adaptive_test(
    user_id,
    skill_id,
    initial_mastery,
    question_df,
    interaction_df,
    n_questions=5,
    simulated_answers=None
):
    mastery = initial_mastery
    previous_correct = None

    attempted = set(
        interaction_df[
            interaction_df["user_id"] == user_id
        ]["problem_id"]
    )

    session = []

    for i in range(n_questions):

        question = select_adaptive_question(
            user_id=user_id,
            skill_id=skill_id,
            mastery=mastery,
            previous_correct=previous_correct,
            question_df=question_df[
                ~question_df["problem_id"].isin(
                    [x["problem_id"] for x in session]
                )
            ],
            interaction_df=interaction_df
        )

        if question is None:
            break

        mastery_before = mastery

        # Simulated student response
        if simulated_answers is not None:
            correct = simulated_answers[i]
        else:
            correct = np.random.choice([0, 1])

        # Update BKT
        mastery = bkt_update(
            mastery,
            correct,
            p_transit=0.20,
            p_guess=0.30,
            p_slip=0.10
        )

        session.append({
            "question_number": i + 1,
            "problem_id": question["problem_id"],
            "skill_name": question["skill_name"],
            "target_difficulty": question["target_difficulty"],
            "question_difficulty": question["difficulty"],
            "difficulty_gap": question["difficulty_gap"],
            "correct": correct,
            "mastery_before": mastery_before,
            "mastery_after": mastery
        })

        previous_correct = correct

    return pd.DataFrame(session)

In [ ]:
top_row = weakness_table.iloc[0]

session_result = run_adaptive_test(
    user_id=top_row["user_id"],
    skill_id=top_row["skill_id"],
    initial_mastery=top_row["bkt_mastery"],
    question_df=question_difficulty,
    interaction_df=bkt_single,
    n_questions=5,
    simulated_answers=[1, 0, 1, 1, 0]
)

session_result

,question_number,problem_id,skill_name,target_difficulty,question_difficulty,difficulty_gap,correct,mastery_before,mastery_after
0,1,91644,Volume Cylinder,0.133333,0.120229,0.013104,1,0.233333,0.581818
1,2,75728,Volume Cylinder,0.681818,0.678571,0.003247,0,0.581818,0.332642
2,3,89771,Volume Cylinder,0.232642,0.237918,0.005276,1,0.332642,0.679403
3,4,89768,Volume Cylinder,0.779403,0.741840,0.037563,1,0.679403,0.891268
4,5,89787,Volume Cylinder,0.950000,0.920000,0.030000,0,0.891268,0.631504


In [ ]:
id="n8x4kp"
avg_gap = session_result["difficulty_gap"].mean()

session_accuracy = session_result["correct"].mean()

initial_mastery = session_result["mastery_before"].iloc[0]
final_mastery = session_result["mastery_after"].iloc[-1]

mastery_change = final_mastery - initial_mastery

print("Adaptive Test Evaluation")
print("-" * 30)

print(f"Average difficulty gap : {avg_gap:.4f}")
print(f"Session accuracy       : {session_accuracy:.2%}")
print(f"Initial mastery        : {initial_mastery:.4f}")
print(f"Final mastery          : {final_mastery:.4f}")
print(f"Mastery change         : {mastery_change:+.4f}")

Adaptive Test Evaluation
------------------------------
Average difficulty gap : 0.0178
Session accuracy       : 60.00%
Initial mastery        : 0.2333
Final mastery          : 0.6315
Mastery change         : +0.3982


**forgetting/recency model**

In [ ]:
print("Order ID range:")
print(bkt_single["order_id"].min())
print(bkt_single["order_id"].max())

print("\nUnique order IDs:")
print(bkt_single["order_id"].nunique())

print("\nSample ordered interactions:")
print(
    bkt_single[
        ["user_id", "skill_id", "order_id", "correct"]
    ]
    .sort_values(["user_id", "order_id"])
    .head(10)
)

Order ID range:
20224085
38310197

Unique order IDs:
227800

Sample ordered interactions:
    user_id  skill_id  order_id  correct
28       14        13  21617893        0
29       14        15  21617915        1
30       14        15  21617924        1
31       14        13  21617931        0
32       14        15  21617942        1
33       14        13  21617953        0
34       14        13  21617962        0
49    21825       279  23944780        1
50    21825       279  23944797        1
51    21825       279  23944811        1


**Interaction-gap based forgetting**

In [ ]:
data = bkt_single.sort_values(
    ["user_id", "order_id"]
).copy()

# Interaction number for each student
data["interaction_number"] = (
    data.groupby("user_id").cumcount()
)

# Last interaction number where this skill was practiced
data["last_skill_interaction"] = (
    data.groupby(["user_id", "skill_id"])["interaction_number"]
    .transform("max")
)

# Gap since last practice
data["interaction_gap"] = (
    data["interaction_number"]
    - data["last_skill_interaction"]
)

print(data[
    [
        "user_id",
        "skill_id",
        "order_id",
        "interaction_number",
        "last_skill_interaction",
        "interaction_gap"
    ]
].tail(10))

        user_id  skill_id  order_id  interaction_number  \
446834    96299        46  38214237                 317   
446835    96299        46  38214238                 318   
446836    96299        46  38214239                 319   
446837    96299        46  38214240                 320   
446838    96299        46  38214241                 321   
446839    96299        58  38214250                 322   
446840    96299        58  38214251                 323   
446841    96299        58  38214252                 324   
446842    96299       309  38214256                 325   
446843    96299        39  38214257                 326   

        last_skill_interaction  interaction_gap  
446834                     321               -4  
446835                     321               -3  
446836                     321               -2  
446837                     321               -1  
446838                     321                0  
446839                     324               -2  


**Correct interaction gap**

In [ ]:
data = bkt_single.sort_values(
    ["user_id", "order_id"]
).copy()

# Interaction number for each student
data["interaction_number"] = (
    data.groupby("user_id").cumcount()
)

# Previous interaction number for the same student-skill
data["previous_skill_interaction"] = (
    data.groupby(["user_id", "skill_id"])["interaction_number"]
    .shift(1)
)

# Number of interactions since previous practice of this skill
data["interaction_gap"] = (
    data["interaction_number"]
    - data["previous_skill_interaction"]
)

print(
    data[
        [
            "user_id",
            "skill_id",
            "interaction_number",
            "previous_skill_interaction",
            "interaction_gap"
        ]
    ]
    .dropna()
    .head(20)
)

    user_id  skill_id  interaction_number  previous_skill_interaction  \
30       14        15                   2                         1.0   
31       14        13                   3                         0.0   
32       14        15                   4                         2.0   
33       14        13                   5                         3.0   
34       14        13                   6                         5.0   
50    21825       279                   1                         0.0   
51    21825       279                   2                         1.0   
52    21825       279                   3                         2.0   
53    21825       279                   4                         3.0   
54    21825       279                   5                         4.0   
55    21825       279                   6                         5.0   
56    21825       279                   7                         6.0   
57    21825       279                   8          

**Forgetting score**

In [ ]:
import numpy as np

# Use only repeated skill interactions
recency_df = data[
    data["interaction_gap"].notna()
].copy()

# Forgetting / recency risk
# Larger gap -> larger forgetting score
decay_rate = 0.05

recency_df["forgetting_score"] = (
    1 - np.exp(
        -decay_rate * recency_df["interaction_gap"]
    )
)

print(
    recency_df[
        [
            "user_id",
            "skill_id",
            "interaction_number",
            "interaction_gap",
            "forgetting_score"
        ]
    ].head(20)
)

    user_id  skill_id  interaction_number  interaction_gap  forgetting_score
30       14        15                   2              1.0          0.048771
31       14        13                   3              3.0          0.139292
32       14        15                   4              2.0          0.095163
33       14        13                   5              2.0          0.095163
34       14        13                   6              1.0          0.048771
50    21825       279                   1              1.0          0.048771
51    21825       279                   2              1.0          0.048771
52    21825       279                   3              1.0          0.048771
53    21825       279                   4              1.0          0.048771
54    21825       279                   5              1.0          0.048771
55    21825       279                   6              1.0          0.048771
56    21825       279                   7              1.0          0.048771

**Current mastery ko forgetting ke saath adjust karna**

In [ ]:
# Get latest interaction for every student-skill pair
latest_recency = (
    recency_df
    .sort_values(["user_id", "skill_id", "interaction_number"])
    .groupby(["user_id", "skill_id"])
    .tail(1)
    [
        [
            "user_id",
            "skill_id",
            "interaction_gap",
            "forgetting_score"
        ]
    ]
)

# Combine with final BKT mastery
learner_state = mastery_table.merge(
    latest_recency,
    on=["user_id", "skill_id"],
    how="left"
)

# No forgetting information -> assume no forgetting penalty
learner_state["forgetting_score"] = (
    learner_state["forgetting_score"]
    .fillna(0)
)

# Effective mastery after accounting for forgetting
learner_state["effective_mastery"] = (
    learner_state["bkt_mastery"]
    * (1 - learner_state["forgetting_score"])
)

print(
    learner_state[
        [
            "user_id",
            "skill_id",
            "skill_name",
            "bkt_mastery",
            "interaction_gap",
            "forgetting_score",
            "effective_mastery"
        ]
    ]
    .head(20)
)

    user_id  skill_id                                     skill_name  \
0        14        13                                         Median   
1        14        15                                          Range   
2     21825       279           Multiplication and Division Integers   
3     51950       311            Equation Solving Two or Fewer Steps   
4     52613       278     Addition and Subtraction Positive Decimals   
5     52613        86                                      Exponents   
6     52613       340                          Distributive Property   
7     53167       278     Addition and Subtraction Positive Decimals   
8     53167       277              Addition and Subtraction Integers   
9     53167       279           Multiplication and Division Integers   
10    53167        24                                     Congruence   
11    53167        54                                       Rounding   
12    53167        67                       Multiplication Fract

**Forgetting model ko realistic banana**

In [ ]:
decay_rate = 0.01

recency_df["forgetting_score_v2"] = (
    1 - np.exp(
        -decay_rate * recency_df["interaction_gap"]
    )
)

latest_recency = (
    recency_df
    .sort_values(
        ["user_id", "skill_id", "interaction_number"]
    )
    .groupby(["user_id", "skill_id"])
    .tail(1)
    [
        [
            "user_id",
            "skill_id",
            "interaction_gap",
            "forgetting_score_v2"
        ]
    ]
)

learner_state = mastery_table.merge(
    latest_recency,
    on=["user_id", "skill_id"],
    how="left"
)

learner_state["forgetting_score_v2"] = (
    learner_state["forgetting_score_v2"]
    .fillna(0)
)

learner_state["effective_mastery"] = (
    learner_state["bkt_mastery"]
    * (1 - learner_state["forgetting_score_v2"])
)

print(
    learner_state[
        [
            "user_id",
            "skill_id",
            "skill_name",
            "bkt_mastery",
            "interaction_gap",
            "forgetting_score_v2",
            "effective_mastery"
        ]
    ]
    .head(20)
)

    user_id  skill_id                                     skill_name  \
0        14        13                                         Median   
1        14        15                                          Range   
2     21825       279           Multiplication and Division Integers   
3     51950       311            Equation Solving Two or Fewer Steps   
4     52613       278     Addition and Subtraction Positive Decimals   
5     52613        86                                      Exponents   
6     52613       340                          Distributive Property   
7     53167       278     Addition and Subtraction Positive Decimals   
8     53167       277              Addition and Subtraction Integers   
9     53167       279           Multiplication and Division Integers   
10    53167        24                                     Congruence   
11    53167        54                                       Rounding   
12    53167        67                       Multiplication Fract

**Final learner weakness score**

In [ ]:
# Mastery weakness after accounting for forgetting
learner_state["mastery_weakness"] = (
    1 - learner_state["effective_mastery"]
)

# Recent performance
recent_performance = (
    bkt_single
    .sort_values(["user_id", "skill_id", "order_id"])
    .groupby(["user_id", "skill_id"])
    .tail(5)
    .groupby(["user_id", "skill_id"])["correct"]
    .mean()
    .reset_index(name="recent_accuracy")
)

learner_state = learner_state.merge(
    recent_performance,
    on=["user_id", "skill_id"],
    how="left"
)

learner_state["recent_weakness"] = (
    1 - learner_state["recent_accuracy"]
)

# Combined learner weakness
learner_state["weakness_score"] = (
    0.55 * learner_state["mastery_weakness"]
    + 0.25 * learner_state["recent_weakness"]
    + 0.20 * learner_state["forgetting_score_v2"]
)

learner_state = learner_state.sort_values(
    "weakness_score",
    ascending=False
)

print(
    learner_state[
        [
            "user_id",
            "skill_id",
            "skill_name",
            "bkt_mastery",
            "effective_mastery",
            "recent_accuracy",
            "forgetting_score_v2",
            "weakness_score"
        ]
    ].head(20)
)

       user_id  skill_id                                      skill_name  \
2231     75361        77                                Finding Percents   
7734     79016       368                          Solving for a variable   
23015    96243        42                          Perimeter of a Polygon   
15899    84572        82                             Scientific Notation   
22332    96226       301                  Surface Area Rectangular Prism   
965      70746       322        Write Linear Equation from Ordered Pairs   
927      70746       308                                   Volume Sphere   
23071    96244        92                                Pattern Finding    
7672     79015        67                        Multiplication Fractions   
17058    85694       350             Solving Systems of Linear Equations   
8411     79032        21  Interior Angles Figures with More than 3 Sides   
1075     70759         4                     Histogram as Table or Graph   
15413    841

**Turn weakness into actionable recommendations**

In [ ]:
def get_learning_priority(row):

    mastery = row["effective_mastery"]
    recent = row["recent_accuracy"]
    weakness = row["weakness_score"]

    if mastery < 0.25 and recent <= 0.20:
        priority = "Critical"
        action = "Relearn concept from basics"

    elif mastery < 0.50 and recent <= 0.40:
        priority = "High"
        action = "Review concept and practice more"

    elif mastery < 0.70:
        priority = "Medium"
        action = "Practice targeted questions"

    else:
        priority = "Low"
        action = "Continue normal practice"

    return pd.Series({
        "priority": priority,
        "recommended_action": action
    })


priority_info = learner_state.apply(
    get_learning_priority,
    axis=1
)

learner_state = pd.concat(
    [learner_state, priority_info],
    axis=1
)

recommendation_table = learner_state[
    [
        "user_id",
        "skill_id",
        "skill_name",
        "bkt_mastery",
        "effective_mastery",
        "recent_accuracy",
        "forgetting_score_v2",
        "weakness_score",
        "priority",
        "recommended_action"
    ]
].sort_values(
    "weakness_score",
    ascending=False
)

recommendation_table.head(20)

,user_id,skill_id,skill_name,bkt_mastery,effective_mastery,recent_accuracy,forgetting_score_v2,weakness_score,priority,recommended_action
2231,75361,77,Finding Percents,0.235655,0.023864,0.000000,0.898734,0.966622,Critical,Relearn concept from basics
7734,79016,368,Solving for a variable,0.238495,0.007059,0.250000,0.970401,0.927697,High,Review concept and practice more
23015,96243,42,Perimeter of a Polygon,0.581901,0.029262,0.200000,0.949713,0.923848,Critical,Relearn concept from basics
15899,84572,82,Scientific Notation,0.233333,0.054188,0.000000,0.767764,0.923749,Critical,Relearn concept from basics
22332,96226,301,Surface Area Rectangular Prism,0.581819,0.041519,0.200000,0.928639,0.912892,Critical,Relearn concept from basics
965,70746,322,Write Linear Equation from Ordered Pairs,0.581833,0.054390,0.200000,0.906519,0.901389,Critical,Relearn concept from basics
927,70746,308,Volume Sphere,0.323837,0.002411,0.400000,0.992553,0.897184,High,Review concept and practice more
23071,96244,92,Pattern Finding,0.394230,0.012024,0.400000,0.969499,0.887286,High,Review concept and practice more
7672,79015,67,Multiplication Fractions,0.339193,0.027020,0.333333,0.920341,0.885874,High,Review concept and practice more
17058,85694,350,Solving Systems of Linear Equations,0.233333,0.081652,0.000000,0.650062,0.885104,Critical,Relearn concept from basics


**Recommendation → Actual Questions**

In [ ]:
def get_student_recommendation(
    user_id,
    recommendation_df,
    question_df,
    interaction_df,
    n_questions=5
):
    # Student ki recommendations
    student_recs = recommendation_df[
        recommendation_df["user_id"] == user_id
    ].copy()

    if len(student_recs) == 0:
        return None, None

    # Highest weakness first
    top = student_recs.sort_values(
        "weakness_score",
        ascending=False
    ).iloc[0]

    skill_id = top["skill_id"]
    mastery = top["effective_mastery"]

    # Already attempted questions
    attempted = set(
        interaction_df[
            interaction_df["user_id"] == user_id
        ]["problem_id"]
    )

    candidates = question_df[
        question_df["skill_id"] == skill_id
    ].copy()

    candidates = candidates[
        ~candidates["problem_id"].isin(attempted)
    ].copy()

    if len(candidates) == 0:
        return top, pd.DataFrame()

    # Target difficulty slightly below current effective mastery
    target = max(0.05, mastery - 0.10)

    candidates["difficulty_gap"] = (
        candidates["difficulty"] - target
    ).abs()

    selected = candidates.sort_values(
        "difficulty_gap"
    ).head(n_questions)

    return top, selected

In [ ]:
top_rec, recommended_questions = get_student_recommendation(
    user_id=75361,
    recommendation_df=recommendation_table,
    question_df=question_difficulty,
    interaction_df=bkt_single,
    n_questions=5
)

print("TOP RECOMMENDATION")
print(
    top_rec[
        [
            "skill_name",
            "weakness_score",
            "priority",
            "recommended_action",
            "effective_mastery"
        ]
    ]
)

print("\nRECOMMENDED QUESTIONS")
print(
    recommended_questions[
        [
            "problem_id",
            "skill_name",
            "difficulty",
            "difficulty_gap"
        ]
    ]
)

TOP RECOMMENDATION
skill_name                       Finding Percents
weakness_score                           0.966622
priority                                 Critical
recommended_action    Relearn concept from basics
effective_mastery                        0.023864
Name: 2231, dtype: object

RECOMMENDED QUESTIONS
       problem_id        skill_name  difficulty  difficulty_gap
12836      148030  Finding Percents    0.050000    4.163336e-17
12842      148036  Finding Percents    0.071429    2.142857e-02
12768      147922  Finding Percents    0.076923    2.692308e-02
12755      147909  Finding Percents    0.090909    4.090909e-02
12764      147918  Finding Percents    0.090909    4.090909e-02
